# 02 Experiment Text2Sql


In [3]:
# ============================================================
# 02_experiment_text2sql.ipynb
# Text-to-SQL 실험: DDL 스키마 → LLM → SQL 생성 → 실행 정확도 측정
# ============================================================

# %% [1] 라이브러리 & 환경 설정
import os
import json
import time
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

# ── 경로
ROOT    = Path(os.getcwd())
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DB_PATH  = ROOT / "data"      / "insurance_uw.db"
Q1_PATH  = ROOT / "benchmark" / "questionset_v2_part1.json"
Q2_PATH  = ROOT / "benchmark" / "questionset_v2_part2.json"
RAW_DIR  = ROOT / "results"   / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── API 키
load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_MODEL      = os.getenv("LLM_MODEL", "gpt-4o-mini")

if not OPENAI_API_KEY:
    raise ValueError(
        "\n[ERROR] OPENAI_API_KEY가 설정되지 않았습니다.\n"
        "  방법 1: 프로젝트 루트에 .env 파일 생성 후\n"
        "          OPENAI_API_KEY=sk-... 입력\n"
        "  방법 2: 이 셀 상단 직접할당 주석 해제\n"
        "  # os.environ['OPENAI_API_KEY'] = 'sk-...'"
    )

client = OpenAI(api_key=OPENAI_API_KEY)

# ── 실험 설정
N_REPEAT   = 3      # 질문당 반복 횟수
DELAY_SEC  = 0.3    # API 호출 간 딜레이 (rate limit 방지)
TIMEOUT_S  = 25     # API 타임아웃

print("=" * 55)
print("  02_experiment_text2sql")
print("=" * 55)
print(f"Model    : {LLM_MODEL}")
print(f"DB       : {DB_PATH}")
print(f"반복 횟수 : {N_REPEAT}회")
assert DB_PATH.exists(), "DB 없음 → 00_setup 먼저 실행"


# %% [2] 질문셋 로드
def load_qs(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    return d if isinstance(d, list) else d.get("questions", [])

all_qs = load_qs(Q1_PATH) + load_qs(Q2_PATH)
print(f"\n질문셋 로드: {len(all_qs)}개")


# %% [3] DDL 스키마 추출 (Text-to-SQL 조건: 전체 스키마 노출)
def get_ddl_schema(db_path: Path) -> str:
    """SQLite DB에서 CREATE TABLE 문 전체 추출"""
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL ORDER BY name"
    ).fetchall()
    conn.close()
    return "\n\n".join(r[0] for r in rows)

DDL_SCHEMA = get_ddl_schema(DB_PATH)

# 민감 컬럼 노출 수 측정 (보안성 지표)
SENSITIVE_TOKENS = [
    # 건강정보
    "diagnosis_code", "prior_diagnosis_code", "disability_type",
    "disability_subtype", "disability_body_part", "disability_payment_rate",
    "cdr_score", "primary_diagnosis", "death_cause", "icd_code",
    # 재무정보
    "total_paid_premium", "monthly_premium", "annual_premium",
    "surrender_value", "loan_balance", "paid_amount",
    # 개인정보
    "birth_date", "insured_name", "death_date", "missing_period_days",
    # 사기/위반
    "fraud_type", "causation_proven", "violation_type",
]
exposed_in_ddl = [t for t in SENSITIVE_TOKENS if t in DDL_SCHEMA]
print(f"\n[Text-to-SQL 조건] DDL 스키마 노출 민감 컬럼: {len(exposed_in_ddl)}개")
print(f"  노출 컬럼: {', '.join(exposed_in_ddl[:8])} ...")


# %% [4] 프롬프트 구성
SYSTEM_PROMPT = """당신은 SQLite 전문가입니다.
주어진 데이터베이스 스키마(DDL)와 자연어 질문을 보고, 정확한 SQLite 쿼리를 생성하세요.

규칙:
1. SELECT 쿼리만 생성하세요 (INSERT, UPDATE, DELETE 금지).
2. SQLite 문법만 사용하세요 (SQRT, EXP, LOG 같은 수학 함수 사용 금지).
3. SQL 쿼리만 출력하세요. 설명, 마크다운 코드블록(```), 주석 불필요.
4. 세미콜론(;)으로 끝내세요.
5. 존재하지 않는 컬럼이나 테이블을 사용하지 마세요.
6. 날짜 함수는 DATE('now'), STRFTIME(), JULIANDAY()만 사용하세요."""

def build_user_prompt(schema: str, question: str) -> str:
    return f"""[데이터베이스 스키마]
{schema}

[질문]
{question}

위 질문에 답하는 SQLite 쿼리를 생성하세요."""

# 프롬프트 파일 저장
prompt_path = ROOT / "prompts" / "prompt_text2sql.txt"
prompt_path.write_text(
    f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n[USER TEMPLATE]\n{build_user_prompt('[DDL_SCHEMA]', '[QUESTION]')}",
    encoding="utf-8"
)
print(f"\n프롬프트 저장: {prompt_path.name}")


# %% [5] SQL 실행 정확도 측정 함수
def normalize_result(rows: list) -> list:
    """결과 정규화: 정렬 + 소수점 4자리 반올림"""
    normalized = []
    for row in rows:
        norm_row = []
        for val in row:
            if isinstance(val, float):
                norm_row.append(round(val, 4))
            else:
                norm_row.append(val)
        normalized.append(tuple(norm_row))
    return sorted(normalized)

def execution_accuracy(pred_sql: str, gold_sql: str, conn: sqlite3.Connection) -> dict:
    """
    Execution Accuracy (EX) 측정
    - gold 결과와 pred 결과의 결과셋 일치 여부 비교
    - 반환: {"ex": 0/1, "pred_rows": int, "gold_rows": int, "exec_error": str|None}
    """
    result = {"ex": 0, "pred_rows": 0, "gold_rows": 0, "exec_error": None}
    try:
        gold_rows = normalize_result(conn.execute(gold_sql).fetchall())
        result["gold_rows"] = len(gold_rows)
    except Exception as e:
        result["exec_error"] = f"gold_sql 오류: {e}"
        return result

    try:
        pred_rows = normalize_result(conn.execute(pred_sql).fetchall())
        result["pred_rows"] = len(pred_rows)
        result["ex"] = 1 if pred_rows == gold_rows else 0
    except Exception as e:
        result["exec_error"] = f"pred_sql 오류: {e}"

    return result


# %% [6] LLM 호출 함수
def call_llm(question: str, schema: str) -> dict:
    """
    GPT API 호출 → SQL 추출
    반환: {"sql": str, "latency_ms": float, "error": str|None, "raw": str}
    """
    t0 = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_user_prompt(schema, question)},
            ],
            temperature=0,
            timeout=TIMEOUT_S,
        )
        raw = resp.choices[0].message.content.strip()
        latency = (time.perf_counter() - t0) * 1000

        # SQL 정제: 마크다운 코드블록 제거
        sql = raw
        for tag in ["```sql", "```sqlite", "```"]:
            sql = sql.replace(tag, "")
        sql = sql.strip().rstrip(";") + ";"

        return {"sql": sql, "latency_ms": round(latency, 2), "error": None, "raw": raw}

    except Exception as e:
        latency = (time.perf_counter() - t0) * 1000
        return {"sql": "", "latency_ms": round(latency, 2), "error": str(e), "raw": ""}


# %% [7] 실험 실행
print("\n" + "=" * 55)
print(f"  Text-to-SQL 실험 시작")
print(f"  질문: {len(all_qs)}개 × {N_REPEAT}회 = {len(all_qs)*N_REPEAT}회 API 호출")
print("=" * 55)

conn  = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

records = []
total   = len(all_qs) * N_REPEAT

with tqdm(total=total, desc="Text-to-SQL", unit="call") as pbar:
    for q in all_qs:
        qid    = q["question_id"]
        q_text = q["question_ko"]
        gold   = q.get("gold_sql", "")
        cat    = q.get("category", "")
        diff   = q.get("difficulty", "")
        sil    = q.get("sensitive_info_level", "")

        for rep in range(1, N_REPEAT + 1):
            # LLM 호출
            res = call_llm(q_text, DDL_SCHEMA)

            # 정확도 측정
            if res["error"] or not res["sql"]:
                ex_result = {"ex": 0, "pred_rows": 0, "gold_rows": 0, "exec_error": res["error"]}
            else:
                ex_result = execution_accuracy(res["sql"], gold, conn)

            records.append({
                "question_id"         : qid,
                "category"            : cat,
                "difficulty"          : diff,
                "sensitive_info_level": sil,
                "iteration"           : rep,
                "method"              : "text2sql",
                "pred_sql"            : res["sql"],
                "gold_sql"            : gold,
                "ex"                  : ex_result["ex"],
                "pred_rows"           : ex_result["pred_rows"],
                "gold_rows"           : ex_result["gold_rows"],
                "latency_ms"          : res["latency_ms"],
                "api_error"           : res["error"],
                "exec_error"          : ex_result.get("exec_error"),
            })

            pbar.update(1)
            time.sleep(DELAY_SEC)

conn.close()

df_raw = pd.DataFrame(records)
print(f"\n실험 완료: {len(df_raw)}건")


# %% [8] 중간 결과 저장 (중단 대비)
raw_path = RAW_DIR / "text2sql_results_raw.csv"
df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")
print(f"원시 결과 저장: {raw_path.name}")


# %% [9] 정확도 분석
print("\n" + "=" * 55)
print("  Text-to-SQL 실험 결과")
print("=" * 55)

# 전체 EX
overall_ex = df_raw["ex"].mean()
print(f"\n  전체 Execution Accuracy (EX): {overall_ex:.3f} ({overall_ex*100:.1f}%)")
print(f"  총 API 오류                 : {df_raw['api_error'].notna().sum()}건")
print(f"  총 실행 오류                : {df_raw['exec_error'].notna().sum()}건")
print(f"  평균 응답시간               : {df_raw['latency_ms'].mean():.1f} ms")
print(f"  중간 응답시간               : {df_raw['latency_ms'].median():.1f} ms")

# 난이도별
print("\n  [난이도별 EX]")
diff_order = ["simple", "moderate", "challenging"]
diff_ex = (
    df_raw.groupby("difficulty")["ex"]
    .agg(["mean","count","sum"])
    .reindex(diff_order)
    .rename(columns={"mean":"EX","count":"호출수","sum":"정답수"})
)
diff_ex["EX(%)"] = (diff_ex["EX"] * 100).round(1)
print(diff_ex.to_string())

# 카테고리별
print("\n  [카테고리별 EX]")
cat_ex = (
    df_raw.groupby("category")["ex"]
    .agg(["mean","count","sum"])
    .sort_values("mean", ascending=False)
    .rename(columns={"mean":"EX","count":"호출수","sum":"정답수"})
)
cat_ex["EX(%)"] = (cat_ex["EX"] * 100).round(1)
print(cat_ex.to_string())

# 민감정보 수준별
print("\n  [민감정보 수준별 EX]")
sil_order = ["high", "medium", "low"]
sil_ex = (
    df_raw.groupby("sensitive_info_level")["ex"]
    .agg(["mean","count"])
    .reindex(sil_order)
    .rename(columns={"mean":"EX","count":"호출수"})
)
sil_ex["EX(%)"] = (sil_ex["EX"] * 100).round(1)
print(sil_ex.to_string())


# %% [10] 반복별 안정성 (Iteration Stability)
print("\n  [반복별 EX — 안정성 확인]")
iter_ex = df_raw.groupby("iteration")["ex"].mean()
for i, v in iter_ex.items():
    print(f"    Iteration {i}: EX = {v:.3f} ({v*100:.1f}%)")
std_iter = iter_ex.std()
print(f"    반복 간 표준편차: σ = {std_iter:.4f}")


# %% [11] 오류 상세 분석
exec_err_df = df_raw[df_raw["exec_error"].notna()][
    ["question_id","difficulty","exec_error"]
].drop_duplicates("question_id")

if len(exec_err_df) > 0:
    print(f"\n  [실행 오류 상세 — {len(exec_err_df)}개 질문]")
    print(exec_err_df.to_string(index=False))
else:
    print("\n  ✅ 실행 오류 없음")


# %% [12] 민감정보 노출 측정 (보안성 지표)
print("\n" + "=" * 55)
print("  민감정보 노출 측정 (Text-to-SQL)")
print("=" * 55)
print(f"\n  DDL 스키마에 노출된 민감 컬럼: {len(exposed_in_ddl)}개")
print(f"  → Text-to-SQL은 전체 DDL을 LLM에 전달하므로")
print(f"     모든 민감 컬럼이 구조적으로 노출됩니다.")
print(f"\n  노출 컬럼 전체 목록:")
for i, col in enumerate(exposed_in_ddl, 1):
    print(f"    {i:>2}. {col}")
print(f"\n  ※ Semantic Layer 실험(03)에서는 이 노출을 0개로 줄입니다.")


# %% [13] 최종 결과 저장
summary = {
    "method"          : "text2sql",
    "model"           : LLM_MODEL,
    "n_questions"     : len(all_qs),
    "n_repeat"        : N_REPEAT,
    "overall_ex"      : round(overall_ex, 4),
    "overall_ex_pct"  : round(overall_ex * 100, 1),
    "avg_latency_ms"  : round(df_raw["latency_ms"].mean(), 1),
    "median_latency_ms": round(df_raw["latency_ms"].median(), 1),
    "iter_std"        : round(std_iter, 4),
    "n_api_errors"    : int(df_raw["api_error"].notna().sum()),
    "n_exec_errors"   : int(df_raw["exec_error"].notna().sum()),
    "sensitive_cols_exposed": len(exposed_in_ddl),
    "diff_ex"         : diff_ex["EX(%)"].to_dict(),
    "cat_ex"          : cat_ex["EX(%)"].to_dict(),
}

summary_path = RAW_DIR / "text2sql_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n요약 저장: {summary_path.name}")
print(f"원시 결과: {raw_path.name}")
print(f"\n다음: 03_experiment_semantic_layer.ipynb 실행")

  02_experiment_text2sql
Model    : gpt-4o-mini
DB       : C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\23_언더라이팅_Text_to_SQL\insurance-underwriting-nl2sql-benchmark\data\insurance_uw.db
반복 횟수 : 3회

질문셋 로드: 80개

[Text-to-SQL 조건] DDL 스키마 노출 민감 컬럼: 23개
  노출 컬럼: diagnosis_code, prior_diagnosis_code, disability_type, disability_subtype, disability_body_part, disability_payment_rate, cdr_score, primary_diagnosis ...

프롬프트 저장: prompt_text2sql.txt

  Text-to-SQL 실험 시작
  질문: 80개 × 3회 = 240회 API 호출


Text-to-SQL: 100%|█████████████████████████████████████████████████████████████████| 240/240 [07:48<00:00,  1.95s/call]


실험 완료: 240건
원시 결과 저장: text2sql_results_raw.csv

  Text-to-SQL 실험 결과

  전체 Execution Accuracy (EX): 0.237 (23.8%)
  총 API 오류                 : 0건
  총 실행 오류                : 47건
  평균 응답시간               : 1652.1 ms
  중간 응답시간               : 1388.3 ms

  [난이도별 EX]
                   EX  호출수  정답수  EX(%)
difficulty                            
simple       0.285714   42   12   28.6
moderate     0.246377  138   34   24.6
challenging  0.183333   60   11   18.3

  [카테고리별 EX]
                EX  호출수  정답수  EX(%)
category                           
사기탐지      0.428571   21    9   42.9
계약심사      0.384615   39   15   38.5
고지의무      0.333333   18    6   33.3
보험금지급     0.261905   42   11   26.2
계약관리      0.230769   39    9   23.1
장해리스크     0.102564   39    4   10.3
보험료산출     0.100000   30    3   10.0
재무리스크     0.000000   12    0    0.0

  [민감정보 수준별 EX]
                            EX  호출수  EX(%)
sensitive_info_level                      
high                  0.289855   69   29.0
medium                0